# DENSS Vendor Update Workflow

<!-- Follows NOTEBOOK_CONVENTIONS.md (ai-context-standard) -->

Shared human/AI workflow for syncing the vendored [DENSS](https://github.com/tdgrant1/denss) copy
in `molass/SAXS/denss/` from the sibling `denss` git repository. Replaces the old manual process
described in `update-memo.txt` (`update-denss.py` / `update-custom-codes.py`), which relied on
fuzzy `diff_match_patch` merges against upstream paths (`bin/`, `saxstats/`) that no longer match
the current `denss` repo layout.

**How to use this notebook**: run the cells top to bottom together with the AI. Diagnostic cells
(diffs, greps) are read-only and safe to re-run any time. Cells that write to vendored files are
explicitly marked and gated behind a `CONFIRM_APPLY` flag — review the diff output above them
before flipping it to `True`.

See [`denss/UPSTREAM.md`](../denss/UPSTREAM.md) for the `# molass-fork:` marker convention this
workflow depends on, and `molass-library/Copilot/workflow_notes.md` for downstream ripple areas
(RgEstimator, SDM/SdmEstimator) that have needed follow-up fixes after past denss version bumps.

## Tool gotchas (learned the hard way, 2026-09-16 sync)

- The sibling `denss` repo (`C:\Users\takahashi\GitHub\denss`) is **outside this VS Code
  workspace**. `grep_search`/`semantic_search` silently return empty or wrong results there —
  that's a tool-scope limitation, not confirmation that something doesn't exist upstream. Use
  `run_in_terminal` (`Select-String`) or `read_file` with the absolute path instead.
- A `# molass-fork:` marker only marks the line it sits on, not an edit's full footprint. E.g.
  `estimate_Vp_etal`'s marker covers its qmax-floor line, but the `self.logger` it *reads* is
  *set* in `Sasrec.__init__` — a separate, unmarked location. When reapplying a marked edit,
  grep the whole file for every name it touches, not just the marked line. Cell [5b] below is
  an independent (different-marker) re-check specifically to catch this class of miss.
- `compile(merged_src, ...)` (cell [5c]) only catches syntax errors, not missing-attribute bugs.
  Those only surface by actually exercising the code path — run the full pytest suite **and**
  `tests-ipynb/generic/910_DenssUtils.ipynb` manually before considering a sync done.
- If a notebook's kernel already imported `molass.SAXS.denss.core` before a fix lands on disk,
  re-running a cell alone won't pick it up (module is cached in `sys.modules`). Either restart
  that kernel, or run `importlib.reload(molass.SAXS.denss.core)` +
  `importlib.reload(molass.SAXS.denss)` in it.
- `aicEditNotebookCell` defaults to the "active" notebook when `notebookUri` is omitted — with
  multiple notebooks open, always pass `notebookUri` explicitly or an edit can silently land in
  the wrong file (surfaces as a confusing "cellNumber out of range" error).


In [4]:
# [1] Setup: locate sibling denss repo and vendored copy
import os
import re
import subprocess
import difflib
import pathlib

REPO_ROOT = pathlib.Path(r"c:\Users\takahashi\GitHub\molass-library")
DENSS_VENDORED = REPO_ROOT / "molass" / "SAXS" / "denss"
DENSS_UPSTREAM = pathlib.Path(r"c:\Users\takahashi\GitHub\denss")

assert DENSS_VENDORED.exists(), DENSS_VENDORED
assert DENSS_UPSTREAM.exists(), DENSS_UPSTREAM


def run(cmd, cwd):
    return subprocess.run(cmd, cwd=cwd, capture_output=True, text=True, encoding="utf-8", check=True).stdout


def read_text(path):
    return pathlib.Path(path).read_text(encoding="utf-8")


def write_text(path, text):
    pathlib.Path(path).write_text(text, encoding="utf-8", newline="\n")


print("Vendored version file:", read_text(DENSS_VENDORED / "_version.py").strip())
print("Upstream working copy version file:", read_text(DENSS_UPSTREAM / "denss" / "_version.py").strip())
print("Upstream HEAD:", run(["git", "log", "-1", "--oneline"], cwd=DENSS_UPSTREAM).strip())


Vendored version file: __version__ = "1.8.7"
Upstream working copy version file: __version__ = "1.8.8"
Upstream HEAD: 5009b1c update version number to 1.8.8


## Step 1 — What changed upstream since the last vendored sync

`LAST_SYNCED_TAG` records the upstream tag that `denss/_version.py` currently reflects. Update it
once a sync is completed and committed.


In [23]:
# [2] Diff upstream from the last vendored tag to HEAD
LAST_SYNCED_TAG = "5009b1c"  # updated 2026-09-16: synced to this commit (v1.8.8, not yet tagged upstream)

print(run(["git", "diff", "--stat", LAST_SYNCED_TAG, "HEAD"], cwd=DENSS_UPSTREAM))
print(run(["git", "log", "--oneline", f"{LAST_SYNCED_TAG}..HEAD"], cwd=DENSS_UPSTREAM))


## Step 2 — Current local fork edits (must survive the sync)

These are the deviations from pristine upstream that make molass's copy work (import path, GUI
progress hooks, NumPy 2.0 compatibility, etc.). Cross-check the list below against the "Summary
of local edits" section in `UPSTREAM.md` — if they've drifted apart, `UPSTREAM.md` is stale.


In [5]:
# [3] List current `# molass-fork:` markers in the vendored files
for name in ["core.py", "options.py"]:
    text = read_text(DENSS_VENDORED / name)
    hits = [f"  L{i + 1}: {line.strip()}" for i, line in enumerate(text.splitlines()) if "molass-fork" in line]
    print(f"{name}: {len(hits)} marker(s)")
    print("\n".join(hits))


core.py: 4 marker(s)
  L313: # molass-fork: explicit float() to satisfy stricter struct.pack() in Python 3.14
  L888: # molass-fork: np.trapz removed in NumPy 2.0; np.trapezoid is the replacement
  L894: # molass-fork: np.trapz removed in NumPy 2.0; np.trapezoid is the replacement
  L1207: # molass-fork: np.in1d removed in NumPy 2.0; np.isin is the replacement
options.py: 0 marker(s)



## Step 2b — Confirmed update scope across `molass/SAXS`

Cross-checked which files under `molass/SAXS` actually import the vendored denss package, to
answer "what is expected to change" beyond the `denss/` folder itself.

**A. Vendored files** (synced directly from upstream this pass): `core.py`/`core-orig.py`,
`options.py`/`options-orig.py` (unchanged upstream this round), `_version.py`,
`scripts/denss_fit_data.py`, `scripts/denss_pdb2mrc.py`, `UPSTREAM.md`.

**B. Downstream files that call into the vendored code** (not synced themselves, but review/adapt
after the sync — cell [2c] below verifies this list stays accurate):

| File | Why it's in scope |
|---|---|
| `DenssUtils.py` | `fit_data_impl` calls `denss.estimate_dmax(...)` directly |
| `DmaxEstimation.py` | hand-copied "copy & modify" of the *old* `estimate_dmax` — most affected by the new BIC estimator |
| `DenssDetector.py` | `from molass.SAXS.denss.core import *` (wildcard import) |
| `DenssTools.py` | imports `reconstruct_abinitio_from_scattering_profile` |

**C. Not in scope**: `DetectorInfo.py`, `FourierIllust.py`, `MrcViewer.py`, `Models/`, `Theory/`,
`Simulator.py` — no direct import of the vendored `denss` package.

> **Correction**: an earlier pass claimed `reconstruct_abinitio_from_scattering_profile` doesn't
> exist upstream at all — that was a false negative from `grep_search`, which only searches the
> open workspace folders and can't see the sibling `denss` repo (not a workspace root). Verified
> via terminal (`Select-String` on the absolute path): the function **does** exist upstream;
> molass adds `progress_cb`/`gui` hooks to it (`UPSTREAM.md` item #2), and separately adds a
> `gui` param + `qmax = max(0.1, qmax)` floor to `optimize_alpha` (item #3). Both are structural
> edits, not 1-line replacements, so `CORE_FORK_EDITS` doesn't cover them — Step 4's markdown
> already says to reapply those manually. Cell [5b] below checks whether they survived the merge.
>
> **Tool lesson**: for anything in the sibling `denss` repo (outside this workspace), use
> `run_in_terminal` (`Select-String`) or `read_text()` with the absolute path — not `grep_search`.



In [9]:
# [2c] Verify: which molass/SAXS files import the vendored denss package
import re as _re

saxs_dir = REPO_ROOT / "molass" / "SAXS"
pattern = _re.compile(r"from molass\.SAXS\.denss|from \.denss|import molass\.SAXS\.denss as denss")

dependents = []
for path in saxs_dir.rglob("*.py"):
    rel = path.relative_to(saxs_dir)
    if rel.parts[0] in ("denss", "denss-update"):
        continue
    if pattern.search(read_text(path)):
        dependents.append(str(rel))

print("molass/SAXS files depending on the vendored denss package:")
for p in sorted(dependents):
    print(" ", p)


molass/SAXS files depending on the vendored denss package:
  DenssDetector.py
  DenssTools.py
  DenssUtils.py
  DmaxEstimation.py


## Step 3 — Preview the merge: structural upstream changes vs. the old baseline

Diffing the *pristine* baseline we saved last time (`core-orig.py`) against the current upstream
file shows only what upstream changed — with no local edits mixed in.


In [6]:
# [4] Diff current -orig.py baseline against upstream HEAD
def upstream_file_lines(rel):
    return read_text(DENSS_UPSTREAM / "denss" / rel).splitlines(keepends=True)


upstream_new_source = {}  # cached for later cells

for orig_name, upstream_rel in [("core-orig.py", "core.py"), ("options-orig.py", "options.py")]:
    old_lines = read_text(DENSS_VENDORED / orig_name).splitlines(keepends=True)
    new_lines = upstream_file_lines(upstream_rel)
    upstream_new_source[upstream_rel] = "".join(new_lines)

    diff = list(difflib.unified_diff(old_lines, new_lines, fromfile=orig_name, tofile=f"upstream/{upstream_rel}", n=2))
    print(f"=== {orig_name} vs upstream {upstream_rel}: {len(diff)} diff lines ===")
    print("".join(diff[:200]))  # preview only, first ~200 lines


=== core-orig.py vs upstream core.py: 737 diff lines ===
--- core-orig.py
+++ upstream/core.py
@@ -1,5 +1,3 @@
 #!/usr/bin/env python
-# molass-fork: DO NOT EDIT --- pristine upstream baseline (DENSS v1.8.7) for diffing
-# molass-fork: against the working copy (core.py). See UPSTREAM.md.
 #
 #    core.py
@@ -813,4 +811,90 @@
     return Iq[(~np.isclose(Iq[:, 1], 0)) & (~np.isclose(Iq[:, 2], 0))]
 
+def clean_low_q_artifacts(q, I, err, window_size=10, z_threshold=3.0, slope_limit_neg=-5.0, slope_limit_pos=2.0,
+                          buffer=4):
+    """
+    Scans backwards to trim beam stop artifacts.
+
+    Args:
+        window_size: Increased to 10 to be 'stiffer' against smooth artifacts.
+        z_threshold: Decreased to 3.0 to be slightly stricter on outliers.
+        slope_limit_neg: Rejects sharp UPTURNS (Leakage).
+        slope_limit_pos: Rejects sharp DOWNTURNS (Shadowing).
+        buffer: If a cut is found, discard this many EXTRA points to be safe.
+    """
+    # Sa

## Step 4 — Reapply known fork edits as exact literal patches

Unlike `diff_match_patch`'s fuzzy positional matching, these are exact `str.replace` patches on
text we author ourselves — so a mismatch is a loud `MISSING` entry, not a silent wrong-location
patch. Keep `CORE_FORK_EDITS` in sync with the "Summary of local edits" list in `UPSTREAM.md`.
Multi-line/structural edits (GUI progress hooks, `optimize_alpha` floor) aren't 1-line replacements
— reapply those manually with the AI's file-editing tools, using the diff in Step 3 as the guide.


In [25]:
# [5] Reapply known fork edits onto the new upstream source
# Exact-text edits. Keep in sync with denss/UPSTREAM.md "Summary of local edits".
# 4th field = expected occurrence count (np.trapz hits 3 sites per UPSTREAM.md; everything
# else is a single location). The structural entries span more than one line because upstream
# rewrote their enclosing functions (`reconstruct_abinitio_from_scattering_profile` kept its
# signature/body shape; `Sasrec.estimate_Vp_etal` kept its shape; `Sasrec.optimize_alpha` was
# fully rewritten with a new L-curve algorithm, so the `gui` hook is re-homed onto the new
# progress-print line rather than the old one).
CORE_FORK_EDITS = [
    ("import path adjustment", "from denss.resources import", "from molass.SAXS.denss.resources import", 1),
    ("np.trapz -> np.trapezoid", "np.trapz(", "np.trapezoid(", 3),
    ("np.in1d -> np.isin", "np.in1d(", "np.isin(", 1),
    (
        "write_mrc float() conversion (marker at old L313 - previously missed!)",
        "    side = np.atleast_1d(side)\n    if len(side) == 1:\n        a, b, c = side, side, side\n    elif len(side) == 3:\n        a, b, c = side\n    else:",
        "    side = np.atleast_1d(side)\n    # molass-fork: explicit float() to satisfy stricter struct.pack() in Python 3.14\n    # (upstream passes 1-element numpy arrays directly to struct.pack('<fff', ...))\n    if len(side) == 1:\n        a, b, c = float(side[0]), float(side[0]), float(side[0])\n    elif len(side) == 3:\n        a, b, c = float(side[0]), float(side[1]), float(side[2])\n    else:",
        1,
    ),
    (
        "progress_cb param (reconstruct_abinitio_from_scattering_profile)",
        "path='.', gui=False, DENSS_GPU=False):",
        "path='.', gui=False, DENSS_GPU=False, progress_cb=None):",
        1,
    ),
    (
        "progress_cb call site (reconstruct_abinitio_from_scattering_profile)",
        "        rho = newrho\n\n    # convert back to numpy outside of for loop",
        "        rho = newrho\n\n        if progress_cb is not None:\n            progress_cb(j, chi[j], rg[j], supportV[j])\n\n    # convert back to numpy outside of for loop",
        1,
    ),
    (
        "Sasrec.__init__ self.logger attribute (previously missed! caught by AttributeError"
        " at runtime in tests-ipynb/generic/910_DenssUtils.ipynb - estimate_Vp_etal reads"
        " self.logger but nothing was setting it)",
        "class Sasrec(object):\n    def __init__(self, Iq, D, qc=None, r=None, nr=None, alpha=0.0, ne=2, extrapolate=True):\n        self.Iq = Iq",
        "class Sasrec(object):\n    def __init__(self, Iq, D, qc=None, r=None, nr=None, alpha=0.0, ne=2, extrapolate=True):\n        debug = False\n        if debug:\n            self.logger = logging.getLogger(__name__)\n        else:\n            self.logger = None\n        self.Iq = Iq",
        1,
    ),
    (
        "qmax floor + logger.info (Sasrec.estimate_Vp_etal)",
        "        qmax = 8. / self.rg\n        if np.isnan(qmax):\n            qmax = 8. / (self.D / 3.5)\n        Iq = np.vstack((self.q, self.I, self.Ierr)).T\n        sasrec4vp = Sasrec(Iq[self.q < qmax], self.D, alpha=self.alpha * oversmoothing, extrapolate=self.extrapolation)",
        "        qmax = 8. / self.rg\n        if np.isnan(qmax):\n            qmax = 8. / (self.D / 3.5)\n        qmax = max(0.1, qmax)       # temporary fix to avoid too low qmax values \n        if self.logger is not None:\n            self.logger.info(\"Estimating Porod volume with oversmoothing factor %.1e and qmax = %.3f A^-1\" % (oversmoothing, qmax))\n        Iq = np.vstack((self.q, self.I, self.Ierr)).T\n        sasrec4vp = Sasrec(Iq[self.q < qmax], self.D, alpha=self.alpha * oversmoothing, extrapolate=self.extrapolation)",
        1,
    ),
    (
        "gui param (Sasrec.optimize_alpha, re-homed onto new algorithm)",
        "    def optimize_alpha(self, quiet=False):",
        "    def optimize_alpha(self, quiet=False, gui=False):",
        1,
    ),
    (
        "gui-routed progress print (Sasrec.optimize_alpha, re-homed onto new algorithm)",
        "        min_extrap_list, extrap_rough_list = [], []\n\n        for i, alpha_exp in enumerate(alphas):\n            if not quiet and i % 10 == 0:\n                sys.stdout.write(f\"\\rScanning Alpha... {i / len(alphas):.0%}\")\n                sys.stdout.flush()",
        "        min_extrap_list, extrap_rough_list = [], []\n\n        if gui:\n            my_logger = logging.getLogger()\n        for i, alpha_exp in enumerate(alphas):\n            if not quiet and i % 10 == 0:\n                msg = f\"Scanning Alpha... {i / len(alphas):.0%}\"\n                if gui:\n                    my_logger.info(msg)\n                else:\n                    sys.stdout.write(f\"\\r{msg}\")\n                    sys.stdout.flush()",
        1,
    ),
]


def apply_known_edits(source: str, edits):
    applied, missing, mismatched = [], [], []
    for desc, old, new, expect in edits:
        count = source.count(old)
        if count == 0:
            missing.append(desc)
        elif count != expect:
            mismatched.append((desc, f"expected {expect}, found {count}"))
        else:
            source = source.replace(old, new)
            applied.append(desc)
    return source, applied, missing, mismatched


merged_core_src, applied, missing, mismatched = apply_known_edits(upstream_new_source["core.py"], CORE_FORK_EDITS)
print("applied:", applied)
print("MISSING (need manual attention, e.g. renamed/removed upstream):", missing)
print("MISMATCHED COUNT (anchor found but not the expected number of times):", mismatched)


applied: ['import path adjustment', 'np.trapz -> np.trapezoid', 'np.in1d -> np.isin', 'write_mrc float() conversion (marker at old L313 - previously missed!)', 'progress_cb param (reconstruct_abinitio_from_scattering_profile)', 'progress_cb call site (reconstruct_abinitio_from_scattering_profile)', 'Sasrec.__init__ self.logger attribute (previously missed! caught by AttributeError at runtime in tests-ipynb/generic/910_DenssUtils.ipynb - estimate_Vp_etal reads self.logger but nothing was setting it)', 'qmax floor + logger.info (Sasrec.estimate_Vp_etal)', 'gui param (Sasrec.optimize_alpha, re-homed onto new algorithm)', 'gui-routed progress print (Sasrec.optimize_alpha, re-homed onto new algorithm)']
MISSING (need manual attention, e.g. renamed/removed upstream): []
MISMATCHED COUNT (anchor found but not the expected number of times): []


In [26]:
# [5c] Sanity check: does the merged source even parse as valid Python?
compile(merged_core_src, "core.py (merged, not yet written)", "exec")
print("merged_core_src compiles cleanly:", len(merged_core_src), "chars")


merged_core_src compiles cleanly: 245135 chars


In [29]:
# [5b] Independent verification that structural fork edits survived the merge, using
# different marker strings than CORE_FORK_EDITS itself (defense in depth: if CORE_FORK_EDITS
# silently fails to apply for some reason, this still catches it before CONFIRM_APPLY).
# Extend this list whenever a new multi-line/structural edit is added to CORE_FORK_EDITS.
# Lesson from the 2026-09-16 sync: two edits (write_mrc float conversion, and the self.logger
# assignment in Sasrec.__init__) were found only by a runtime AttributeError in
# tests-ipynb/generic/910_DenssUtils.ipynb, AFTER CONFIRM_APPLY had already written files to
# disk - compile() alone does not catch a missing attribute.
STRUCTURAL_EDIT_MARKERS = [
    ("progress_cb hook in reconstruct_abinitio_from_scattering_profile", "progress_cb"),
    ("qmax floor in Sasrec.estimate_Vp_etal", "qmax = max(0.1, qmax)"),
    ("self.logger SET in Sasrec.__init__ (the 'write' site for estimate_Vp_etal's 'read')", "debug = False\n        if debug:\n            self.logger"),
    ("write_mrc float() conversion", "float(side[0]), float(side[1]), float(side[2])"),
]

_results = []
for desc, marker in STRUCTURAL_EDIT_MARKERS:
    in_current = marker in current_core_src
    in_merged = marker in merged_core_src
    ok = (not in_current) or in_merged
    _results.append(ok)
    status = "OK" if ok else "MISSING - must be reapplied before CONFIRM_APPLY"
    print(f"{desc}: in current core.py={in_current}, in merged_core_src={in_merged}  [{status}]")

ALL_STRUCTURAL_EDITS_OK = all(_results)
print("\nALL_STRUCTURAL_EDITS_OK =", ALL_STRUCTURAL_EDITS_OK)


progress_cb hook in reconstruct_abinitio_from_scattering_profile: in current core.py=True, in merged_core_src=True  [OK]
qmax floor in Sasrec.estimate_Vp_etal: in current core.py=True, in merged_core_src=True  [OK]
self.logger SET in Sasrec.__init__ (the 'write' site for estimate_Vp_etal's 'read'): in current core.py=False, in merged_core_src=True  [OK]
write_mrc float() conversion: in current core.py=True, in merged_core_src=True  [OK]

ALL_STRUCTURAL_EDITS_OK = True


## Step 5 — Apply (writes vendored files) — requires confirmation

Only run this after the human and AI have reviewed Steps 3–4 together and manually reapplied any
structural edits into `merged_core_src`. Flip `CONFIRM_APPLY` to `True` to write.


In [ ]:
# [6] Apply: write new core-orig.py, options-orig.py, and merged core.py (guarded)
CONFIRM_APPLY = True  # reviewed diffs + edits with the human; ready to write

NEW_UPSTREAM_TAG = "v1.8.8"  # matches the version bump in Step 7; not yet tagged upstream as of this sync


def with_orig_header(desc, source):
    header = (
        f"# molass-fork: DO NOT EDIT --- pristine upstream baseline (DENSS {NEW_UPSTREAM_TAG}) for diffing\n"
        f"# molass-fork: against the working copy ({desc}). See UPSTREAM.md.\n"
    )
    lines = source.splitlines(keepends=True)
    return lines[0] + header + "".join(lines[1:]) if lines and lines[0].startswith("#!") else header + source


if CONFIRM_APPLY and not globals().get("ALL_STRUCTURAL_EDITS_OK", True):
    raise RuntimeError(
        "Cell [5b]'s independent structural-edit check found a MISSING edit. "
        "Fix CORE_FORK_EDITS, re-run cells [5]-[5c], and confirm ALL_STRUCTURAL_EDITS_OK is "
        "True before applying. (This guard exists because exactly this kind of miss reached "
        "disk undetected during the 2026-09-16 sync.)"
    )

if CONFIRM_APPLY:
    write_text(DENSS_VENDORED / "core-orig.py", with_orig_header("core.py", upstream_new_source["core.py"]))
    write_text(DENSS_VENDORED / "options-orig.py", with_orig_header("options.py", upstream_new_source["options.py"]))
    write_text(DENSS_VENDORED / "core.py", merged_core_src)
    print("Wrote core-orig.py, options-orig.py, core.py. Remaining manual work:", missing)
else:
    print("Skipped (CONFIRM_APPLY is False). Review the diffs above first.")


Wrote core-orig.py, options-orig.py, core.py. Remaining manual work: []


## Step 6 — Scripts sync (`denss_fit_data.py`, `denss_pdb2mrc.py`)

`denss_pdb2mrc.py` is invoked as a subprocess by `DenssUtils.run_pdb2mrc`; `denss_fit_data.py`'s
logic is not — `DenssUtils.fit_data_impl` calls `denss.estimate_dmax` in `core.py` directly. Diff
both anyway, since drift here is easy to miss.


In [8]:
# [7] Diff vendored scripts against upstream
for name in ["denss_fit_data.py", "denss_pdb2mrc.py"]:
    old_lines = read_text(DENSS_VENDORED / "scripts" / name).splitlines(keepends=True)
    new_lines = read_text(DENSS_UPSTREAM / "denss" / "scripts" / name).splitlines(keepends=True)
    diff = list(difflib.unified_diff(old_lines, new_lines, fromfile=f"vendored/{name}", tofile=f"upstream/{name}", n=2))
    print(f"=== {name}: {len(diff)} diff lines ===")
    print("".join(diff))


=== denss_fit_data.py: 36 diff lines ===
--- vendored/denss_fit_data.py
+++ upstream/denss_fit_data.py
@@ -102,5 +102,6 @@
 
     if args.n1 is None:
-        n1 = 0
+        # n1 = 0
+        n1 = denss.clean_low_q_artifacts(Iq[:,0], Iq[:,1], Iq[:,2], window_size=6, z_threshold=5.0)
     else:
         n1 = args.n1
@@ -117,5 +118,5 @@
         #the high q data, even though by default
         #denss.Sasrec does extrapolate.
-        D, sasrec = denss.estimate_dmax(Iq, clean_up=True)
+        D, sasrec = denss.estimate_dmax(Iq[n1:n2], clean_up=True)
     else:
         D = args.dmax
@@ -163,18 +164,4 @@
     print("Number of experimental Shannon channels: %d"%(nsh))
     print("Number of calculated Shannon channels: %d"%(nshc))
-    if (nsh > 500) or (nshc>500):
-        print("WARNING: Nsh > 500. Calculation may take a while. Please double check Dmax is accurate.")
-        #give the user a few seconds to cancel with CTRL-C
-        waittime = 10
-        try:
-            for i in ra

In [21]:
# [7b] Apply: overwrite vendored scripts wholesale (confirmed no local "molass" edits present)
CONFIRM_APPLY_SCRIPTS = True

if CONFIRM_APPLY_SCRIPTS:
    for name in ["denss_fit_data.py", "denss_pdb2mrc.py"]:
        write_text(DENSS_VENDORED / "scripts" / name, read_text(DENSS_UPSTREAM / "denss" / "scripts" / name))
    print("Wrote scripts/denss_fit_data.py, scripts/denss_pdb2mrc.py")
else:
    print("Skipped (CONFIRM_APPLY_SCRIPTS is False).")


Wrote scripts/denss_fit_data.py, scripts/denss_pdb2mrc.py


In [22]:
# [7c] Apply: bump _version.py
write_text(DENSS_VENDORED / "_version.py", '__version__ = "1.8.8"\n')
print("Vendored version now:", read_text(DENSS_VENDORED / "_version.py").strip())


Vendored version now: __version__ = "1.8.8"


## Step 7 — Version bump, `UPSTREAM.md`, and test

Manual checklist (deliberately not automated — each item needs a judgment call):

1. Update `_version.py` and the "Vendored version" line in `UPSTREAM.md`.
2. Update `LAST_SYNCED_TAG` in Step 1 above.
3. Refresh the "Summary of local edits" list in `UPSTREAM.md` if `CORE_FORK_EDITS` changed.
4. Decide separately whether to port the new upstream auto Dmax/alpha BIC estimator into
   `DmaxEstimation.py` — that's a design change, not a vendor sync; track it as its own item.
5. Check the downstream ripple areas from the last bump: `RgEstimator.py`, `SDM.py`,
   `SdmEstimator.py` (see `Copilot/workflow_notes.md`, "Temporary fix for DENSS 1.8.7").
6. Run the pytest cell below, **and** manually run `tests-ipynb/generic/910_DenssUtils.ipynb`
   end-to-end — it exercises `Sasrec.estimate_Vp_etal` via `molass_legacy.BoundedLRF`, a path
   the pytest suite does not cover, and is exactly where the 2026-09-16 `self.logger` /
   `write_mrc` gaps were actually caught.
7. If `910_DenssUtils.ipynb` (or any other notebook) already has a running kernel that imported
   `molass.SAXS.denss.core` before this sync's fixes landed on disk, restart that kernel (or
   `importlib.reload(molass.SAXS.denss.core)` + `importlib.reload(molass.SAXS.denss)` in it) —
   re-running a cell alone will keep using the stale cached module.


In [28]:
# [8] ⏳ Run the denss-dependent test suite (long-running)
result = subprocess.run(
    ["py", "-m", "pytest", "tests/tutorial/11-rigorous_optimization.py", "-v"],
    cwd=REPO_ROOT, capture_output=True, text=True,
)
print(result.stdout[-4000:])
print(result.stderr[-2000:])
print("Also run tests-ipynb/generic/910_DenssUtils.ipynb manually (not pytest-collected).")


ation.py::test_001_quick_decomposition PASSED [ 25%]
tests/tutorial/11-rigorous_optimization.py::test_002_rigorous_optimization PASSED [ 50%]
tests/tutorial/11-rigorous_optimization.py::test_003_has_rigorous_results PASSED [ 75%]
tests/tutorial/11-rigorous_optimization.py::test_004_wait_for_rigorous_results PASSED [100%]

============================== warnings summary ===============================
tests/tutorial/11-rigorous_optimization.py::test_001_quick_decomposition
  C:\Users\takahashi\GitHub\molass-library\molass\Guinier\RgEstimator.py:32: RuntimeWarning: invalid value encountered in log
    data[nb:ne, 0] ** 2, np.log(data[nb:ne, 1]))

tests/tutorial/11-rigorous_optimization.py::test_001_quick_decomposition
  C:\Users\takahashi\GitHub\molass-library\molass\Guinier\RgEstimator.py:32: RuntimeWarning: divide by zero encountered in log
    data[nb:ne, 0] ** 2, np.log(data[nb:ne, 1]))

tests/tutorial/11-rigorous_optimization.py::test_001_quick_decomposition
  C:\Users\takahashi\Git